In [67]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import os

import scipy.stats
from tqdm import tqdm

from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
import itertools
import time

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F


In [68]:
torch.cuda.empty_cache()

In [69]:
args = {}

# Training params
args['batch_size'] = 5

args['epochs'] = 50*10*4
args['start_epoch'] = 0

# For optimizer
args['lr'] = 1e-4 # LR for B-SNN params
args['lr_decay'] = 1e-6
args['sigma_true'] = 1e-4

args['device'] =  torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [70]:
# Load planet_ids
train_adc_info = pd.read_csv('/mnt/tier2/project/p200475/Ariel/train_adc_info.csv',
                           index_col='planet_id')
# Load sensor data
train_data= np.load('/mnt/tier2/project/p200475/Ariel/data_train_clean.npy')

In [71]:
class CustomDataset(Dataset):
    def __init__(self, data_train, indices, path_labels):
        """
        Args:
            data_train (np.array): Array of shape (673, 187, 282, 32).
            indices (np.array): Array of indices corresponding to the rows in df_labels.
            path_labels (str): Path to the CSV file containing labels.
        """
        self.data_train = torch.tensor(data_train)  # Convert the data to a tensor
        self.data_indices = torch.tensor(indices).long()  # Convert indices to tensor
        self.df_labels = pd.read_csv(path_labels)  # Load df_labels as a pandas DataFrame
        
    def __len__(self):
        # Return the total number of samples
        return len(self.data_indices)

    def __getitem__(self, idx):
        # Get the index from data_indices
        index = self.data_indices[idx].item()  # Convert tensor to Python scalar
        
        # Select the data_train sample corresponding to the index
        data_sample = self.data_train[idx]  # Directly access the data without cloning

        # Select the corresponding label row where the first column matches the index
        label_row = self.df_labels[self.df_labels.iloc[:, 0] == index].iloc[0].values[1:]  # Exclude the first column (index)

        # Convert label_row to a tensor
        label = torch.tensor(label_row).float()

        return data_sample, label

# Example usage
path_ground = '/mnt/tier2/project/p200475/Ariel/train_labels.csv'

# Assuming 'train' and 'train_adc_info' are pre-loaded variables containing the training data and indices.
# For example, train could be an array of shape (673, 187, 282, 32), and train_adc_info.index gives the indices.
custom_dataset = CustomDataset(train_data, train_adc_info.index, path_ground)

# Create a DataLoader to iterate over the dataset
dataloader = DataLoader(custom_dataset, batch_size=args['batch_size'], shuffle=True)  # Pass 'custom_dataset' here

# Iterate through the dataset
for batch_data, batch_labels in dataloader:
    print("Batch Data Shape:", batch_data.shape)  # Should be (batch_size, 187, 282, 32)
    print("Batch Labels Shape:", batch_labels.shape)  # Should be (batch_size, 283)
    break


Batch Data Shape: torch.Size([5, 375, 283])
Batch Labels Shape: torch.Size([5, 283])


In [72]:
class SpectralConv_resnet(nn.Module):
    def __init__(self):
        super(SpectralConv_resnet, self).__init__()
        
        self.model = models.resnet50(pretrained=True)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        self.model.fc = nn.Linear(in_features=2048, out_features=283*2, bias=True)

    
    def forward(self, x):
        # Apply convolutional layer
        x = self.model(x)
        
        # Apply a Sigmoid activation after conv1
        x = torch.sigmoid(x)
        
        return x

In [73]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class SpectralConv_efficient(nn.Module):
    def __init__(self):
        super(SpectralConv_efficient, self).__init__()
        
        # Use EfficientNet (pretrained)
        self.model = models.efficientnet_b0(pretrained=False)
        
        # Adjust the first convolutional layer for 1-channel input instead of 3-channel RGB
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)

        # Adjust the final fully connected layer to the desired output size
        self.model.classifier[1] = nn.Linear(in_features=1280, out_features=283*2, bias=True)

    def forward(self, x):
        # Apply convolutional layers and fully connected layers
        x = self.model(x)
        
        # Apply a Sigmoid activation after conv layers
        x = torch.sigmoid(x)
        
        return x

In [74]:
class SpectralConv_efficient_b3(nn.Module):
    def __init__(self):
        super(SpectralConv_efficient_b3, self).__init__()
        
        # Use EfficientNet-B3 (pretrained)
        self.model = models.efficientnet_b3(pretrained=False)
        
        # Adjust the first convolutional layer for 1-channel input instead of 3-channel RGB
        self.model.features[0][0] = nn.Conv2d(1, 40, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)

        # Adjust the final fully connected layer to the desired output size
        self.model.classifier[1] = nn.Linear(in_features=1536, out_features=283*2, bias=True)

    def forward(self, x):
        # Apply convolutional layers and fully connected layers
        x = self.model(x)
        
        # Apply a Sigmoid activation after conv layers
        x = torch.sigmoid(x)
        
        return x

In [75]:
# Load best model weights from checkpoint file
weights_checkpoint = 'efficientnet_b3_850.pth'
scoring_SpecTrans  = SpectralConv_efficient_b3()

scoring_SpecTrans = scoring_SpecTrans.to(args['device'])
# Load the state dict
state_dict = torch.load(weights_checkpoint, map_location=args['device'])


# Load the modified state dict into the model
scoring_SpecTrans.load_state_dict(state_dict)
scoring_SpecTrans.eval()

/home/users/u101881/.local/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/users/u101881/.local/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/tmp/ipykernel_31312/1509801864.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped t

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 39.50 GiB of which 2.06 MiB is free. Process 31347 has 25.20 GiB memory in use. Including non-PyTorch memory, this process has 14.28 GiB memory in use. Of the allocated memory 13.75 GiB is allocated by PyTorch, and 41.76 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [76]:
img_unsqueeze = batch_data.unsqueeze(1)
light_curve = img_unsqueeze.sum(axis=3)/img_unsqueeze.mean()

output = scoring_SpecTrans(img_unsqueeze.to(torch.float32).to(args['device']))

OutOfMemoryError: CUDA out of memory. Tried to allocate 22.00 MiB. GPU 0 has a total capacity of 39.50 GiB of which 2.06 MiB is free. Process 31347 has 25.20 GiB memory in use. Including non-PyTorch memory, this process has 14.28 GiB memory in use. Of the allocated memory 13.76 GiB is allocated by PyTorch, and 39.74 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
plt.plot(output[18,:283].T.cpu().detach().numpy())
plt.title("Spectrum prediction")
plt.show()

In [ ]:
plt.plot(batch_labels[18].T.cpu().detach().numpy())
plt.title("Spectrum ground truth")
plt.show()